In [33]:
import numpy as np
import pandas as pd


In [34]:
df = pd.read_csv("sachin_emotion_dataset_large.csv")

In [35]:
df.sample(10)

,Text,Label
836,Teri kripa se hi sab sambhav hai. Aaj bhi.,Devotional
490,Bheed mein bhi ek ajeeb sa akelapan hai. Khud ...,Melancholic
714,Toofan mazbooti parakhte hain. Khud se wada hai.,Inspiring
326,Bheed mein bhi ek ajeeb sa akelapan hai. Dil se.,Melancholic
650,"Haar sirf ek mod hai, ant nahi.",Inspiring
837,Tera naam hi mera sahara hai. Aaj bhi.,Devotional
844,Hanuman sa bal de prabhu. Har pal.,Devotional
421,"Dil ko samjhaya bahut, par maana hi nahi.",Melancholic
113,Tera naam likhu to lafz mehek jaate hain. Khud...,Romantic
822,Hanuman sa bal de prabhu. Har pal.,Devotional


In [36]:
df.rename(columns = {"Text" : "text", "Label" : "label"}, inplace = True)

In [37]:
df.head()

,text,label
0,Tum meri saanson ki aadat ban gaye ho.,Romantic
1,Tere bina har khushi adhoori si lagti hai. Khu...,Romantic
2,Teri khamoshi bhi mujhe sab keh jaati hai. Dil...,Romantic
3,Tum saath ho to har darr chhota lagta hai. Har...,Romantic
4,Tere bina har khushi adhoori si lagti hai.,Romantic


In [38]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

In [39]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label Mapping (Kaunsa number kiska hai?):", label_mapping)

Label Mapping (Kaunsa number kiska hai?): {'Devotional': 0, 'Inspiring': 1, 'Melancholic': 2, 'Romantic': 3}


In [40]:
from sklearn.model_selection import train_test_split
train_text, test_text, train_labels, test_labels = train_test_split(
    df["text"].tolist(), df["label_encoded"].tolist(), test_size=0.2, random_state=42
)

In [41]:
print(f"\nTotal lines for Training: {len(train_text)}")
print(f"Total lines for Testing: {len(test_text)}")
print("\nSample Train Text:", train_text[0])
print("Uska Label:", train_labels[0])


Total lines for Training: 800
Total lines for Testing: 200

Sample Train Text: Teri baaton mein ghar jaisa sukoon hai. Hamesha.
Uska Label: 3


In [42]:
import tensorflow as tf
from transformers import AutoTokenizer

In [43]:
model_name = "distilbert-base-multingual"

In [44]:
import sys
print(sys.version)
print(sys.executable)

3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
c:\Users\sachi\anaconda3\envs\poetry_dl\python.exe


In [45]:
# TensorFlow aur Transformers ko bhool jao, ye import karo:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

print("Classic ML Brain load ho raha hai...")

# 1. Pipeline banana (Translator + Brain)
# TfidfVectorizer: Words ko numbers/importance mein badlega
# MultinomialNB: Ek probability wala algorithm jo words count karke emotion batata hai
from sklearn.svm import LinearSVC
# MultinomialNB ko hatakar LinearSVC lagaya hai
# ngram_range=(1, 2) ka matlab hai 1 word aur 2 words ke combos dono dekho
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LinearSVC())

model.fit(train_text, train_labels)
print("Training Complete! 🎉")

# 3. Exam lena (Testing)
# Ab hum test data par iska test lenge ki isne kaisa seekha
predictions = model.predict(test_text)
accuracy = accuracy_score(test_labels, predictions)

print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")

# 4. Ek Live Test karke dekhte hain!
# test_poem = ["Teri palkon main sona hai, teri baaton main khushboo hai, tera saath hai toh zindagi mein rang hai."] # Ye ek nayi poetry hai, jiska emotion hum predict karna chahte hain
test_poem = ["Tu akela nahi hai, tere sath tera hausla hai!"]
predicted_number = model.predict(test_poem)
predicted_emotion = le.inverse_transform(predicted_number) # Number ko wapas text mein badalna
print(f"\nLive Test Poem: {test_poem[0]}")
print(f"Predicted Emotion: {predicted_emotion[0]}")

Classic ML Brain load ho raha hai...
Training Complete! 🎉

Final Test Accuracy: 100.00%

Live Test Poem: Tu akela nahi hai, tere sath tera hausla hai!
Predicted Emotion: Devotional


In [46]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification

print("Brain Load ho raha hai...")
model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased", 
    num_labels=4
)
print("Brain Load ho gaya! 🎉")

Brain Load ho raha hai...


RuntimeError: Failed to import transformers.models.distilbert.modeling_tf_distilbert because of the following error (look up to see its traceback):
module 'tensorflow' has no attribute 'version'

In [ ]:
from transformers import TFAutoModelForSequenceClassification
import tensorflow as tf

# 1. Brain Load karna (With a new Classification Head)
# num_labels=4 kyunki hamare paas 4 emotions hain (Romantic, Melancholic, Inspiring, Devotional)
model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased", 
    num_labels=4
)

# 2. Model ko batana ki seekhna kaise hai (Compilation)
# Optimizer: Adam (Ye sikhne ki speed control karta hai. 5e-5 standard speed hai transformer ke liye)
# Loss: SparseCategoricalCrossentropy (Ye model ko batata hai ki uski galti kitni badi hai)
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

print("\nModel Compile ho gaya! Training shuru kar rahe hain... 🚀")

# 3. The Actual Training (Sikhayi)
# epochs=3 ka matlab hai AI poore 800 examples ko 3 baar padhega revise karne ke liye
history = model.fit(
    train_dataset,
    validation_data=test_dataset, # Har round ke baad ispe exam dega
    epochs=3
)

print("\nTraining Complete! 🎉")

RuntimeError: Failed to import transformers.models.distilbert.modeling_tf_distilbert because of the following error (look up to see its traceback):
module 'tensorflow' has no attribute 'version'